Transformação de dados

In [1]:
import pandas as pd
from pathlib import Path
import os
from camara_deputados.ingestion.data_loader import DataLoader
from camara_deputados.extraction.write import DataWrite


In [3]:
# instâncias

bronze = DataLoader('bronze')
salva= DataWrite()



In [4]:
# criando a dim_deputados
df_deputadoDetalhamento = bronze.carregar_tabela('deputados_detalhamento')

# Mapeia as colunas que irão para camada silver e nomeia
mapeamento = {'id':'id_deputado',
           'nomeCivil':'nom_Nome',
           'sexo':'nom_Sexo',
           'dataNascimento':'dat_DataNascimento',
           'ufNascimento':'nom_UF',
           'municipioNascimento':'nom_MunicipioNatal',
           'escolaridade':'nom_Escolaridade'
}

#Resgata as colunas originais
colunas_origem = list(mapeamento.keys())


#cria a  dim_deputado com as colunas renomeadas
dim_deputado = df_deputadoDetalhamento[colunas_origem].rename(columns=mapeamento)



✅ deputados_detalhamento: 513 linhas, 34 colunas


In [11]:
dim_deputado.sample(5)

,id_deputado,nom_Nome,nom_Sexo,dat_DataNascimento,nom_UF,nom_MunicipioNatal,nom_Escolaridade
484,220684,SEBASTIÃO HENRIQUE DE MEDEIROS,M,1983-01-08,PR,Paranavaí,Superior
339,74158,MÁRIO LÚCIO HERINGER,M,1954-09-30,MG,Manhumirim,Superior
388,178860,PAULO VELLOSO DANTAS AZI,M,1963-01-14,BA,Salvador,Superior
454,204535,SAMIA DE SOUZA BOMFIM,F,1989-08-22,SP,Presidente Prudente,Superior
469,204425,SILVIO SERAFIM COSTA FILHO,M,1982-03-05,PE,Recife,Superior


In [5]:
dim_deputado['dat_DataNascimento']=pd.to_datetime(dim_deputado['dat_DataNascimento'],errors='coerce')

dim_deputado = dim_deputado.sort_values("id_deputado").reset_index(drop=True)



In [30]:
# salvando na silver

salva.save_parquet(dim_deputado, 'dim_s_deputados', layer='gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_deputados/dim_s_deputados.parquet


In [7]:
## Dim_frentes


df_frentes = bronze.carregar_tabela('frentes')

mapeamento_frentes = {'id':'id_frentes',
                      'titulo':'nom_TituloFrente',
                      'idLegislatura':'num_Legislatura'

}

colunas_origem = list(mapeamento_frentes.keys())

dim_frente = df_frentes[colunas_origem].rename(columns=mapeamento_frentes)

✅ frentes: 100 linhas, 5 colunas


In [19]:
dim_frente.sample(5)

,id_frentes,nom_TituloFrente,num_Legislatura
42,55628,Frente Parlamentar em Defesa das Vítimas de Ac...,57
78,54560,"Frente Parlamentar Mista em Defesa da Criança,...",57
80,55571,Frente Parlamentar Mista de Apoio à Emancipaçã...,57
56,55603,Frente Parlamentar em Defesa dos Profissionais...,57
37,55638,Frente Parlamentar Mista da Causa QESA,57


In [31]:
# salva na camada gold_temp 

salva.save_parquet(dim_frente, 'dim_s_frente', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_frente/dim_s_frente.parquet


In [ ]:
# Cria dim_partido na gold_temp

df_partidos = bronze.carregar_tabela('partidos')

mapeamento_partidos = {'id':'id_partido',
                       'sigla':'nom_Sigla',
                       'nome':'nom_NomePartido'}

colunas_origem = list(mapeamento_partidos.keys())

dim_partido = df_partidos[colunas_origem].rename(columns=mapeamento_partidos)

✅ partidos: 15 linhas, 5 colunas


In [11]:
dim_partido.sample(5)

,id_partido,nom_Sigla,nom_NomePartido
1,37905,CIDADANIA,Cidadania
8,36896,PODE,Podemos
11,36832,PSB,Partido Socialista Brasileiro
7,37906,PL,Partido Liberal
4,37901,NOVO,Partido Novo


In [32]:
salva.save_parquet(dim_partido,'dim_s_partido','gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_partido/dim_s_partido.parquet


In [16]:
# Criando a dim_proposicao

df_proposicao = bronze.carregar_tabela('proposicoes_detalhamento')

mapeamento_proposicao = {
    'id':'id_proposicao',
    'codTipo':'cod_Tipo',
    'numero':'num_NumeroProposicao',
    'ano':'num_Ano',
    'ementa':'nom_Ementa',
    'keywords':'nom_Keywords'
}

colunas_origem = list(mapeamento_proposicao.keys())

dim_proposicao = df_proposicao[colunas_origem].rename(columns = mapeamento_proposicao)

✅ proposicoes_detalhamento: 60 linhas, 36 colunas


In [33]:
salva.save_parquet(dim_proposicao,'dim_s_proposicao', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/gold_temp/dim_s_proposicao/dim_s_proposicao.parquet


Criando a dim_tema a partir da tabela proposicoes Temas

In [ ]:
#Carregando a proposicoe_temas

df_tema = bronze.carregar_tabela('proposicoes_temas')

✅ proposicoes_temas: 62 linhas, 5 colunas


In [22]:
dim_tema = (
    df_tema[['codTema', 'tema']]
    .drop_duplicates()
    .dropna(subset=['codTema'])
    .sort_values('codTema')
    .reset_index(drop=True)
)

In [21]:
display(dim_tema)

,codTema,tema
0,34,Administração Pública
1,37,Comunicações
2,40,Economia
3,41,Cidades e Desenvolvimento Urbano
4,42,Direito Civil e Processual Civil
5,43,Direito Penal e Processual Penal
6,44,Direitos Humanos e Minorias
7,46,Educação
8,48,Meio Ambiente e Desenvolvimento Sustentável
9,52,Previdência e Assistência Social


In [ ]:
# renomeando e salvando na silver 
 
mapeamento_temas = {
    'codTema':'cod_Tema',
    'tema':'nom_Tema'
}

colunas_origem = list(mapeamento_temas.keys())

dim_tema = dim_tema[colunas_origem].rename(columns=mapeamento_temas)

salva.save_parquet(dim_tema, 'dim_s_tema', 'gold_temp')

💾 Salvo em: /mnt/d/GabrielaTorres/estudos/Univesp_Projetos/pi_camara_deputados/data/silver/dim_s_tema/dim_s_tema.parquet


In [24]:
def analisar_dfs(dfs):
    resultado = []

    for nome_tabela, df in dfs.items():
        for coluna in df.columns:
            
            total = len(df)
            nulos = df[coluna].isnull().sum()
            
            resultado.append({
                "tabela": nome_tabela,
                "coluna": coluna,
                "tipo": df[coluna].dtype,
                "nulos": nulos,
                "%_nulos": nulos / total,
                "unicos": df[coluna].astype(str).nunique(),
                "total_linhas": total
            })

    return pd.DataFrame(resultado)

In [25]:
dfs_dim = {
    'dim_deputado':dim_deputado,
    'dim_frente':dim_frente,
    'dim_partido':dim_partido,
    'dim_proposicacao':dim_proposicao,
    'dim_tema':dim_tema

}

In [26]:
df_analiseDim = analisar_dfs(dfs_dim)

In [29]:
display(df_analiseDim)

,tabela,coluna,tipo,nulos,%_nulos,unicos,total_linhas,data_extracao
0,dim_deputado,id_deputado,int64,0,0.000000,513,513,2026-04-21 10:21:09.795757
1,dim_deputado,nom_Nome,str,0,0.000000,513,513,2026-04-21 10:21:09.795757
2,dim_deputado,nom_Sexo,str,0,0.000000,2,513,2026-04-21 10:21:09.795757
3,dim_deputado,dat_DataNascimento,datetime64[us],0,0.000000,502,513,2026-04-21 10:21:09.795757
4,dim_deputado,nom_UF,str,1,0.001949,27,513,2026-04-21 10:21:09.795757
5,dim_deputado,nom_MunicipioNatal,str,2,0.003899,267,513,2026-04-21 10:21:09.795757
6,dim_deputado,nom_Escolaridade,str,12,0.023392,11,513,2026-04-21 10:21:09.795757
7,dim_deputado,data_extracao,datetime64[us],0,0.000000,1,513,2026-04-21 10:21:09.795757
8,dim_frente,id_frentes,int64,0,0.000000,100,100,2026-04-21 10:21:09.795757
9,dim_frente,nom_TituloFrente,str,0,0.000000,100,100,2026-04-21 10:21:09.795757
